# Part 1: Scraping de decretos de emergencia
**Grupo 4 Temporada de lluvias de 1 enero 2025 al 31 de mayo 2025**

**1. Descarga e impresión de robot.txt**

In [1]:
# 1. Descarga e impresión de robot.txt

import requests

headers = {"User-agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/154.0.0.0 Safari/537.36"}

robot_gobpe=requests.get("https://www.gob.pe/robots.txt", headers=headers, timeout=30)

print(robot_gobpe.text)

# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



1.a ¿Qué prohíbe el archivo robots.txt?

Prohíbe el acceso a las rutas que incluyan la ruta /admin/ y el parámetro sheet=.

Para el caso en concreto de los bots GPT y OAI-Search se les pide, además, esperar 5 y 2 segundos entre solicitudes, respectivamente.
Mientras que AhrefsBot tiene prohíbido acceder a cualquier página del sitio (Disallow: /). 

1.b ¿La página de búsquedas /búsquedas está permitida?

Sí, en la medida que no incluya el parámetro sheet=.

1.c ¿Por qué conviene hacer pausas entre página y página? 

Si bien el Crawl-delay alcanza únicamente a los bots de GPT y OAI-search, parece indicar que se busca evitar que el sitio se sature con muchas solicitudes y se bloquee.

**2. Búsqueda de un mes de la temporada (Enero 2025)**


In [2]:
# 2. Búsqueda de un mes de la temporada (Enero 2025)

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as PD
import time
import re

In [3]:
PD.set_option("display.max_colwidth", None)

In [4]:
opciones = Options()
# opciones.add_argument("--headless=new")   # descomenta para ocultarlo
opciones.add_argument("--window-size=1400,900")
opciones.add_argument(
    "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
)

navegador = webdriver.Chrome(options=opciones)
print("Navegador abierto")


navegador.get("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-01-2025&hasta=31-01-2025")


WebDriverWait(navegador, 30)
time.sleep(6)

Navegador abierto


In [ ]:
WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "main ul[role='list'] > li > article"))
)
print("Resultados cargados")

Resultados cargados


**3. Número total de resultados (Enero 2025)**


In [6]:
# 3. Número total de resultados (Enero 2025)

texto_total = navegador.find_element(By.CSS_SELECTOR, "main h2").text
print(texto_total)

total_enero = int(re.search(r"\d+", texto_total).group())
print("Total de resultados:", total_enero)

12 Resultados
Total de resultados: 12


**4. Extracción de número, fecha, título y enlace de cada norma (Enero 2025)**


In [ ]:
# 4. Extracción de número, fecha, título y enlace de cada norma (Enero 2025)

espera = WebDriverWait(navegador, 30)
articulos = espera.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "main ul[role='list'] > li > article"))
)

normas = [] #corresponde a enero 2025
for articulo in articulos:
    enlace = articulo.find_element(By.CSS_SELECTOR, "h4 a")
    fecha = articulo.find_element(By.XPATH, ".//span[starts-with(normalize-space(), 'Publicado')]")
    titulo = articulo.find_element(By.CSS_SELECTOR, "p[id$='-description']")

    normas.append({
        "numero_norma": enlace.text.strip(),
        "fecha_publicacion": fecha.text.strip(),
        "titulo": titulo.text.strip(),
        "enlace_norma": enlace.get_attribute("href"),
    })

total_pagina = int(re.search(r"\d+", navegador.find_element(By.CSS_SELECTOR, "main h2").text).group())

print("Normas extraídas     :", len(normas))
print("Total según la página:", total_pagina)

Normas extraídas     : 12
Total según la página: 12


In [8]:
PD.DataFrame(normas)

,numero_norma,fecha_publicacion,titulo,enlace_norma
0,Decreto Supremo N.° 018-2025-PCM,Publicado: 30 de enero de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto,https://www.gob.pe/institucion/pcm/normas-legales/6529837-018-2025-pcm
1,Decreto Supremo N.° 016-2025-PCM,Publicado: 29 de enero de 2025,"Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos y centros poblados de provincias pertenecientes a los departamentos de Ayacucho, Huancavelica, Junín y Cusco",https://www.gob.pe/institucion/pcm/normas-legales/6529813-016-2025-pcm
2,Decreto Supremo N.° 013-2025-PCM,Publicado: 23 de enero de 2025,"Decreto Supremo que modifica el Decreto Supremo N° 003-2025-PCM, Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Trujillo del departamento de La Libertad",https://www.gob.pe/institucion/pcm/normas-legales/6471904-013-2025-pcm
3,Decreto Supremo N.° 012-2025-PCM,Publicado: 22 de enero de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en el Corredor Vial Sur Apurímac-Cusco-Arequipa,https://www.gob.pe/institucion/pcm/normas-legales/6471894-012-2025-pcm
4,Decreto Supremo N.° 011-2025-PCM,Publicado: 22 de enero de 2025,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huep ...",https://www.gob.pe/institucion/pcm/normas-legales/6471880-011-2025-pcm
5,Decreto Supremo N.° 008-2025-PCM,Publicado: 10 de enero de 2025,Declaratoria del Estado de Emergencia,https://www.gob.pe/institucion/pcm/normas-legales/6410307-008-2025-pcm
6,Decreto Supremo N.° 007-2025-PCM,Publicado: 10 de enero de 2025,Declaratoria del Estado de Emergencia,https://www.gob.pe/institucion/pcm/normas-legales/6410247-007-2025-pcm
7,Decreto Supremo N.° 006-2025-PCM,Publicado: 8 de enero de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en varios distritos de las provincias del departamento de Piura por peligro inminente ante déficit hídrico,https://www.gob.pe/institucion/pcm/normas-legales/6385710-006-2025-pcm
8,Decreto Supremo N.° 004-2025-PCM,Publicado: 3 de enero de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Pataz del departamento de La Libertad,https://www.gob.pe/institucion/pcm/normas-legales/6370714-004-2025-pcm
9,Decreto Supremo N.° 003-2025-PCM,Publicado: 3 de enero de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Trujillo del departamento de La Libertad,https://www.gob.pe/institucion/pcm/normas-legales/6370703-003-2025-pcm


In [40]:
navegador.quit()

**5. Extracción de número, fecha, título y enlace de cada norma**


In [ ]:
# El de enero ya ha sido generado, ahora se genera el de febrero. Se repite el proceso para cada mes de la temporada.
#  5.1 Extracción de número, fecha, título y enlace de cada norma (Febrero 2025)

time.sleep(5)  

navegador = webdriver.Chrome(options=opciones)
print("Navegador abierto")

navegador.get("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-02-2025&hasta=28-02-2025")

Navegador abierto


In [11]:
espera = WebDriverWait(navegador, 30)
articulos = espera.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "main ul[role='list'] > li > article"))
)

normas_febrero = []
for articulo in articulos:
    enlace = articulo.find_element(By.CSS_SELECTOR, "h4 a")
    fecha = articulo.find_element(By.XPATH, ".//span[starts-with(normalize-space(), 'Publicado')]")
    titulo = articulo.find_element(By.CSS_SELECTOR, "p[id$='-description']")

    normas_febrero.append({
        "numero_norma": enlace.text.strip(),
        "fecha_publicacion": fecha.text.strip(),
        "titulo": titulo.text.strip(),
        "enlace_norma": enlace.get_attribute("href"),
    })

total_febrero = int(re.search(r"\d+", navegador.find_element(By.CSS_SELECTOR, "main h2").text).group())

print("Normas extraídas     :", len(normas_febrero))
print("Total según la página:", total_febrero)

Normas extraídas     : 4
Total según la página: 4


In [12]:
PD.DataFrame(normas_febrero)

,numero_norma,fecha_publicacion,titulo,enlace_norma
0,Decreto Supremo N.° 025-2025-PCM,Publicado: 26 de febrero de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Zarumilla del departamento de Tumbes,https://www.gob.pe/institucion/pcm/normas-legales/6539147-025-2025-pcm
1,Decreto Supremo N.° 024-2025-PCM,Publicado: 26 de febrero de 2025,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, H ...",https://www.gob.pe/institucion/pcm/normas-legales/6539143-024-2025-pcm
2,Decreto Supremo N.° 022-2025-PCM,Publicado: 19 de febrero de 2025,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Manitea, Kimbiri y Cielo Punco de la provincia de La Convención del departamento de Cusco y en el distrito de Samuga ...",https://www.gob.pe/institucion/pcm/normas-legales/6539254-022-2025-pcm
3,Decreto Supremo N.° 021-2025-PCM,Publicado: 19 de febrero de 2025,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, H ...",https://www.gob.pe/institucion/pcm/normas-legales/6539250-021-2025-pcm


In [ ]:
navegador.quit()

In [14]:
# 5.2 Extracción de número, fecha, título y enlace de cada norma (Marzo 2025)

time.sleep(5)  

navegador = webdriver.Chrome(options=opciones)
print("Navegador abierto")

navegador.get("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-03-2025&hasta=31-03-2025")

Navegador abierto


In [15]:
espera = WebDriverWait(navegador, 30)
articulos = espera.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "main ul[role='list'] > li > article"))
)

normas_marzo = []
for articulo in articulos:
    enlace = articulo.find_element(By.CSS_SELECTOR, "h4 a")
    fecha = articulo.find_element(By.XPATH, ".//span[starts-with(normalize-space(), 'Publicado')]")
    titulo = articulo.find_element(By.CSS_SELECTOR, "p[id$='-description']")

    normas_marzo.append({
        "numero_norma": enlace.text.strip(),
        "fecha_publicacion": fecha.text.strip(),
        "titulo": titulo.text.strip(),
        "enlace_norma": enlace.get_attribute("href"),
    })

total_marzo = int(re.search(r"\d+", navegador.find_element(By.CSS_SELECTOR, "main h2").text).group())

print("Normas extraídas     :", len(normas_marzo))
print("Total según la página:", total_marzo)

Normas extraídas     : 12
Total según la página: 12


In [16]:
PD.DataFrame(normas_marzo)

,numero_norma,fecha_publicacion,titulo,enlace_norma
0,Decreto Supremo N.° 041-2025-PCM,Publicado: 26 de marzo de 2025,"Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos y centros poblados de provincias pertenecientes a los departamentos de Ayacucho, Huancavelica, Junín y Cusco",https://www.gob.pe/institucion/pcm/normas-legales/6709207-041-2025-pcm
1,Decreto Supremo N.° 040-2025-PCM,Publicado: 26 de marzo de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en el Corredor Vial Sur Apurímac - Cusco - Arequipa,https://www.gob.pe/institucion/pcm/normas-legales/6709197-040-2025-pcm
2,Decreto Supremo N.° 036-2025-PCM,Publicado: 19 de marzo de 2025,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huep ...",https://www.gob.pe/institucion/pcm/normas-legales/6608813-036-2025-pcm
3,Decreto Supremo N.° 035-2025-PCM,Publicado: 17 de marzo de 2025,Decreto Supremo que declara el Estado de Emergencia en Lima Metropolitana del departamento de Lima y en la Provincia Constitucional del Callao,https://www.gob.pe/institucion/pcm/normas-legales/6609801-035-2025-pcm
4,Decreto Supremo N.° 034-2025-PCM,Publicado: 12 de marzo de 2025,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, H ...",https://www.gob.pe/institucion/pcm/normas-legales/6608812-034-2025-pcm
5,Decreto Supremo N.° 033-2025-PCM,Publicado: 12 de marzo de 2025,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, H ...",https://www.gob.pe/institucion/pcm/normas-legales/6608811-033-2025-pcm
6,Decreto Supremo N.° 032-2025-PCM,Publicado: 12 de marzo de 2025,"Decreto Supremo que declara el Estado de Emergencia en el sector comprendido entre el Jr. Jauja, Jr. Áncash, Jr. Huánuco, Prolongación Lucanas, Jr. Santa Rosa y Jr. Huanta del Cercado de Lima de la p ...",https://www.gob.pe/institucion/pcm/normas-legales/6608810-032-2025-pcm
7,Decreto Supremo N.° 031-2025-PCM,Publicado: 6 de marzo de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Trujillo del departamento de La Libertad,https://www.gob.pe/institucion/pcm/normas-legales/6595467-031-2025-pcm
8,Decreto Supremo N.° 030-2025-PCM,Publicado: 6 de marzo de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Pataz del departamento de La Libertad,https://www.gob.pe/institucion/pcm/normas-legales/6595466-030-2025-pcm
9,Decreto Supremo N.° 028-2025-PCM,Publicado: 5 de marzo de 2025,"Decreto Supremo que declara el Estado de Emergencia en los distritos de Anco y Unión Progreso de la provincia de La Mar del departamento de Ayacucho y en los distritos de Megantoni, Kumpirushiato, Ec ...",https://www.gob.pe/institucion/pcm/normas-legales/6595463-028-2025-pcm


In [ ]:
navegador.quit()

In [18]:
# 5.3 Extracción de número, fecha, título y enlace de cada norma (Abril 2025)

time.sleep(5)  

navegador = webdriver.Chrome(options=opciones)
print("Navegador abierto")

navegador.get("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-04-2025&hasta=30-04-2025")


Navegador abierto


In [19]:
espera = WebDriverWait(navegador, 30)
articulos = espera.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "main ul[role='list'] > li > article"))
)

normas_abril = []
for articulo in articulos:
    enlace = articulo.find_element(By.CSS_SELECTOR, "h4 a")
    fecha = articulo.find_element(By.XPATH, ".//span[starts-with(normalize-space(), 'Publicado')]")
    titulo = articulo.find_element(By.CSS_SELECTOR, "p[id$='-description']")

    normas_abril.append({
        "numero_norma": enlace.text.strip(),
        "fecha_publicacion": fecha.text.strip(),
        "titulo": titulo.text.strip(),
        "enlace_norma": enlace.get_attribute("href"),
    })

total_abril = int(re.search(r"\d+", navegador.find_element(By.CSS_SELECTOR, "main h2").text).group())

print("Normas extraídas     :", len(normas_abril))
print("Total según la página:", total_abril)

Normas extraídas     : 15
Total según la página: 15


In [20]:
PD.DataFrame(normas_abril)

,numero_norma,fecha_publicacion,titulo,enlace_norma
0,Decreto Supremo N.° 058-2025-PCM,Publicado: 30 de abril de 2025,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Anco y Unión Progreso de la provincia de La Mar del departamento de Ayacucho y en los distritos de Megantoni, Kumpir ...",https://www.gob.pe/institucion/pcm/normas-legales/6747740-058-2025-pcm
1,Decreto Supremo N.° 057-2025-PCM,Publicado: 30 de abril de 2025,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias del departamento de Loreto, por impacto de daños a consecuencia de intensas precipitaciones",https://www.gob.pe/institucion/pcm/normas-legales/6747737-057-2025-pcm
2,Decreto Supremo N.° 056-2025-PCM,Publicado: 30 de abril de 2025,"Decreto Supremo que prorroga el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Áncash, Cusco, Huánuco, La Libertad, Pasco, Piura y Tumbes, por impacto de daños ...",https://www.gob.pe/institucion/pcm/normas-legales/6747728-056-2025-pcm
3,Decreto Supremo N.° 055-2025-PCM,Publicado: 30 de abril de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Zarumilla del departamento de Tumbes,https://www.gob.pe/institucion/pcm/normas-legales/6747717-055-2025-pcm
4,Decreto Supremo N.° 053-2025-PCM,Publicado: 24 de abril de 2025,"Decreto Supremo que prorroga el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Cusco, Piura, Tumbes y Ucayali, por impacto de daños a consecu ...",https://www.gob.pe/institucion/pcm/normas-legales/6960768-053-2025-pcm
5,Decreto Supremo N.° 052-2025-PCM,Publicado: 23 de abril de 2025,Decreto Supremo que declara el Estado de Emergencia en el distrito de Kosñipata de la provincia de Paucartambo del departamento de Cusco y en los distritos de Manu y Fitzcarrald de la provincia de Ma ...,https://www.gob.pe/institucion/pcm/normas-legales/6960760-052-2025-pcm
6,Decreto Supremo N.° 038-2025-PCM,Publicado: 21 de abril de 2025,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, H ...",https://www.gob.pe/institucion/pcm/normas-legales/6718597-038-2025-pcm
7,Decreto Supremo N.° 050-2025-PCM,Publicado: 16 de abril de 2025,"Decreto Supremo que prorroga el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, ...",https://www.gob.pe/institucion/pcm/normas-legales/6725196-050-2025-pcm
8,Decreto Supremo N.° 049-2025-PCM,Publicado: 16 de abril de 2025,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Manitea, Kimbiri y Cielo Punco de la provincia de La Convención del departamento de Cusco y en el distrito de Samuga ...",https://www.gob.pe/institucion/pcm/normas-legales/6725181-049-2025-pcm
9,Decreto Supremo N.° 0046-2025-PCM,Publicado: 13 de abril de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en Lima Metropolitana del departamento de Lima y en la Provincia Constitucional del Callao,https://www.gob.pe/institucion/pcm/normas-legales/6718646-0046-2025-pcm


In [ ]:
navegador.quit()

In [22]:
# 5.4 Extracción de número, fecha, título y enlace de cada norma (Mayo 2025)

time.sleep(5)  

navegador = webdriver.Chrome(options=opciones)
print("Navegador abierto")

navegador.get("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent&desde=01-05-2025&hasta=31-05-2025")

Navegador abierto


In [23]:
espera = WebDriverWait(navegador, 30)
articulos = espera.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "main ul[role='list'] > li > article"))
)

normas_mayo = []
for articulo in articulos:
    enlace = articulo.find_element(By.CSS_SELECTOR, "h4 a")
    fecha = articulo.find_element(By.XPATH, ".//span[starts-with(normalize-space(), 'Publicado')]")
    titulo = articulo.find_element(By.CSS_SELECTOR, "p[id$='-description']")

    normas_mayo.append({
        "numero_norma": enlace.text.strip(),
        "fecha_publicacion": fecha.text.strip(),
        "titulo": titulo.text.strip(),
        "enlace_norma": enlace.get_attribute("href"),
    })

total_mayo = int(re.search(r"\d+", navegador.find_element(By.CSS_SELECTOR, "main h2").text).group())

print("Normas extraídas     :", len(normas_mayo))
print("Total según la página:", total_mayo)

Normas extraídas     : 11
Total según la página: 11


In [24]:
PD.DataFrame(normas_mayo)

,numero_norma,fecha_publicacion,titulo,enlace_norma
0,Decreto Supremo N.° 073-2025-PCM,Publicado: 29 de mayo de 2025,"Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos y centros poblados de provincias pertenecientes a los departamentos de Ayacucho, Huancavelica, Junín y Cusco",https://www.gob.pe/institucion/pcm/normas-legales/6961461-073-2025-pcm
1,Decreto Supremo N.° 072-2025-PCM,Publicado: 28 de mayo de 2025,"Decreto Supremo que prorroga el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Áncash, Ayacucho, Cusco, Lima y Ucayali, por impacto de daños a consecuencia de ...",https://www.gob.pe/institucion/pcm/normas-legales/6961443-072-2025-pcm
2,Decreto Supremo N.° 071-2025-PCM,Publicado: 28 de mayo de 2025,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Ayna, Santa Rosa, Anchihuay y Río Magdalena de la provincia de La Mar del departamento de Ayacucho",https://www.gob.pe/institucion/pcm/normas-legales/6961426-071-2025-pcm
3,Decreto Supremo N.° 068-2025-PCM,Publicado: 21 de mayo de 2025,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la Provincia de Tambopata y en los distritos de Madre de Dios y Huep ...",https://www.gob.pe/institucion/pcm/normas-legales/6961391-068-2025-pcm
4,Decreto Supremo N.° 067-2025-PCM,Publicado: 21 de mayo de 2025,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias del departamento de Loreto, por impacto de daños a consecuencia de intensas precipitaciones",https://www.gob.pe/institucion/pcm/normas-legales/6961380-067-2025-pcm
5,Decreto Supremo N.° 066-2025-PCM,Publicado: 21 de mayo de 2025,"Decreto Supremo que prorroga el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, ...",https://www.gob.pe/institucion/pcm/normas-legales/6961370-066-2025-pcm
6,Decreto Supremo N.° 065-2025-PCM,Publicado: 16 de mayo de 2025,"Decreto Supremo que prorroga el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Áncash, Huánuco, Madre de Dios y Puno, por impacto de daños a consecuencia de in ...",https://www.gob.pe/institucion/pcm/normas-legales/6961356-065-2025-pcm
7,Decreto Supremo N.° 064-2025-PCM,Publicado: 16 de mayo de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en algunos distritos de Lima Metropolitana del departamento de Lima y en la Provincia Constitucional del Callao,https://www.gob.pe/institucion/pcm/normas-legales/6961344-064-2025-pcm
8,Resolución Directoral N.° 121-2025-DG-HSB,Publicado: 15 de mayo de 2025,"Aprobar la Adopción de nueve (09) Guías de Práctica Clínica de diversos nosocomios, para ser aplicadas en el Servicio de Cuidados Críticos de la Mujer - Departamento de Emergencia y Cuidados Crítico ...",https://www.gob.pe/institucion/honadomani-sb/normas-legales/6783538-121-2025-dg-hsb
9,Decreto Supremo N.° 062-2025-PCM,Publicado: 8 de mayo de 2025,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Trujillo del departamento de La Libertad,https://www.gob.pe/institucion/pcm/normas-legales/6961020-062-2025-pcm


In [ ]:
navegador.quit()

**6. Verificación**

In [26]:
# 6. Verificación 

verificacion = PD.DataFrame({
    "mes": ["Enero 2025", "Febrero 2025", "Marzo 2025", "Abril 2025", "Mayo 2025"],
    "resultados_totales": [total_enero, total_febrero, total_marzo, total_abril, total_mayo],
    "resultados_extraidos": [len(normas), len(normas_febrero), len(normas_marzo), len(normas_abril), len(normas_mayo)],
})

print(verificacion.to_string(index=False))

         mes  resultados_totales  resultados_extraidos
  Enero 2025                  12                    12
Febrero 2025                   4                     4
  Marzo 2025                  12                    12
  Abril 2025                  15                    15
   Mayo 2025                  11                    11


**7. Eliminar normas repetidas**

In [ ]:
# 7. Eliminar normas repetidas

df = PD.concat(
    [
        PD.DataFrame(normas),          # corresponde a enero
        PD.DataFrame(normas_febrero),
        PD.DataFrame(normas_marzo),
        PD.DataFrame(normas_abril),
        PD.DataFrame(normas_mayo),
    ],
    ignore_index=True,
)
print("Normas extraídas (total):", len(df))

df = df.drop_duplicates(subset="enlace_norma").reset_index(drop=True)
print("Normas sin repetidos     :", len(df))



Normas extraídas (total): 54
Normas sin repetidos     : 54


**Respuesta:** 
En nuestra temporada, no se constata la existencia de normas repetidas.

**8. Solo normas de la PCM**


In [ ]:

cantidad_inicial = len(df)

df = df[df["enlace_norma"].str.contains("/institucion/pcm/")].reset_index(drop=True)

print("Normas de otras instituciones removidas:", cantidad_inicial - len(df))
print("Normas de la PCM  :", len(df))


# Se verifica la existencia de normas emitidas por entidades distintas de la PCM.



Normas de otras instituciones removidas: 1
Normas de la PCM  : 53


**Respuesta**: Se coló una resolución del Hospital Nacional Docente Madre Niño San Bartolomé que aprueba guías clínicas.

**9. Obtener títulos completos**

In [29]:
import requests
from bs4 import BeautifulSoup
import time

PD.set_option("display.max_colwidth", None)

def titulo_completo(enlace):
    respuesta2 = requests.get(enlace, headers=headers, timeout=30)
    sopa = BeautifulSoup(respuesta2.text, "html.parser")
    descripcion = sopa.find("div", class_="description")
    time.sleep(1) # Pausa 1 segundo entre páginas

         
    if descripcion and descripcion.find("div"):
        return descripcion.find("div").get_text(" ", strip=True)
    else :
        return "No se encontró descripción"


titulos = []
for enlace in df["enlace_norma"]:
    titulos.append(titulo_completo(enlace))

df["titulo_completo"] = titulos
print("Normas sin titulo completo:", (df["titulo_completo"] == "No se encontró descripción").sum())
df[["numero_norma", "titulo_completo"]].head(3)
           
        




Normas sin titulo completo: 0


,numero_norma,titulo_completo
0,Decreto Supremo N.° 018-2025-PCM,Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto
1,Decreto Supremo N.° 016-2025-PCM,"Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos y centros poblados de provincias pertenecientes a los departamentos de Ayacucho, Huancavelica, Junín y Cusco"
2,Decreto Supremo N.° 013-2025-PCM,"Decreto Supremo que modifica el Decreto Supremo N° 003-2025-PCM, Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Trujillo del departamento de La Libertad"


**# 10. Clasificación de normas usando el título completo**

In [30]:
# 10. Clasificación de normas usando el título completo


lista_lluvia = []
lista_tipo = []
lista_motivo = []

for titulo in df["titulo_completo"]:
    t = titulo.lower()   # título en minúsculas

    # es_lluvia: True si contiene lluvia o precipitaciones
    lista_lluvia.append("lluvia" in t or "precipitaciones" in t)

    # Por tipo:  revisamos "prorroga", porque los títulos de prórroga dicen "declarado", y "declarado" contiene la palabra "declara"
    if "prórroga" in t or "prorroga" in t:
        lista_tipo.append("prorroga")
    elif "declara" in t:
        lista_tipo.append("declara")
    else:
        lista_tipo.append("otro")

    # Por motivo
    if "peligro inminente" in t:
        lista_motivo.append("peligro inminente")
    elif "impacto de daños" in t:
        lista_motivo.append("impacto de daños")
    else:
        lista_motivo.append("otro")

df["es_lluvia"] = lista_lluvia
df["tipo"] = lista_tipo
df["motivo"] = lista_motivo

print(df["es_lluvia"].value_counts())
print(df["tipo"].value_counts())
print(df["motivo"].value_counts())


es_lluvia
False    36
True     17
Name: count, dtype: int64
tipo
prorroga    37
declara     16
Name: count, dtype: int64
motivo
otro                 33
impacto de daños     17
peligro inminente     3
Name: count, dtype: int64


In [31]:
#Filas que quedaron como "otro"
PD.set_option("display.max_colwidth", None)

quedaron_otro = (df["tipo"] == "otro") | (df["motivo"] == "otro")
df.loc[quedaron_otro, ["numero_norma", "titulo_completo", "tipo", "motivo", "es_lluvia"]]


,numero_norma,titulo_completo,tipo,motivo,es_lluvia
0,Decreto Supremo N.° 018-2025-PCM,Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto,prorroga,otro,False
1,Decreto Supremo N.° 016-2025-PCM,"Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos y centros poblados de provincias pertenecientes a los departamentos de Ayacucho, Huancavelica, Junín y Cusco",prorroga,otro,False
2,Decreto Supremo N.° 013-2025-PCM,"Decreto Supremo que modifica el Decreto Supremo N° 003-2025-PCM, Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Trujillo del departamento de La Libertad",prorroga,otro,False
3,Decreto Supremo N.° 012-2025-PCM,Decreto Supremo que prorroga el Estado de Emergencia declarado en el Corredor Vial Sur Apurímac-Cusco-Arequipa,prorroga,otro,False
4,Decreto Supremo N.° 011-2025-PCM,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huepetuhe de la provincia de Manu del departamento de Madre de Dios",prorroga,otro,False
5,Decreto Supremo N.° 008-2025-PCM,Declaratoria del Estado de Emergencia,declara,otro,False
6,Decreto Supremo N.° 007-2025-PCM,Declaratoria del Estado de Emergencia,declara,otro,False
8,Decreto Supremo N.° 004-2025-PCM,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Pataz del departamento de La Libertad,prorroga,otro,False
9,Decreto Supremo N.° 003-2025-PCM,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Trujillo del departamento de La Libertad,prorroga,otro,False
10,Decreto Supremo N.° 002-2025-PCM,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Virú del departamento de La Libertad,prorroga,otro,False


**Revisión de filas "Otro"**

Se aprecia que todas son "prorroga" (37) o "declara" (16).

Se revisaron las normas clasificadas como "otro" y se encontró que casi todas estaban bien categorizadas. 
Solo el DS 013-2025-PCM debería estar como "otro", aunque no afecta el análisis porque no está relacionado con lluvias. También se corrigieron los títulos de los DS 007-2025-PCM y 008-2025-PCM usando los PDF, identificando que el primero sí corresponde a un caso de lluvias.

**10.1 Corrección de títulos genéricos "Declaratoria de Estado de Emergencia"**

Al revisar las normas que quedaron como "otro", encontramos que en gob.pe el DS 007-2025-PCM y el DS 008-2025-PCM solo dicen "Declaratoria del Estado de Emergencia", sin lugar ni motivo. Abrimos el PDF de cada decreto, copiamos su título real y volvimos a clasificar con la misma regla del ítem 10.

In [ ]:
# 10.1 Corrección de títulos "DS 007-2025-PCM" y "DS 008-2025-PCM"

titulo_007 = ("Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas "
              "provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, "
              "Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Loreto, "
              "Moquegua, Pasco, Puno, San Martín, Tacna y la Provincia Constitucional del Callao, "
              "por peligro inminente ante intensas precipitaciones pluviales")

titulo_008 = ("Decreto Supremo que declara el Estado de Emergencia en los distritos de Chiclayo, "
              "José Leonardo Ortiz y La Victoria de la provincia de Chiclayo del departamento de "
              "Lambayeque, por peligro inminente ante colapso del sistema de alcantarillado")

df.loc[df["numero_norma"] == "Decreto Supremo N.° 007-2025-PCM", "titulo_completo"] = titulo_007
df.loc[df["numero_norma"] == "Decreto Supremo N.° 008-2025-PCM", "titulo_completo"] = titulo_008

# Volvemos a clasificar con la misma regla del ítem 10, ahora con los títulos corregidos

lista_lluvia = []
lista_tipo = []
lista_motivo = []

for titulo in df["titulo_completo"]:
    t = titulo.lower()   # título en minúsculas

    lista_lluvia.append("lluvia" in t or "precipitaciones" in t)

    if "prórroga" in t or "prorroga" in t:
        lista_tipo.append("prorroga")
    elif "declara" in t:
        lista_tipo.append("declara")
    else:
        lista_tipo.append("otro")

    if "peligro inminente" in t:
        lista_motivo.append("peligro inminente")
    elif "impacto de daños" in t:
        lista_motivo.append("impacto de daños")
    else:
        lista_motivo.append("otro")

df["es_lluvia"] = lista_lluvia
df["tipo"] = lista_tipo
df["motivo"] = lista_motivo

print(df["es_lluvia"].value_counts())
print(df["tipo"].value_counts())
print(df["motivo"].value_counts())

corregidas = df["numero_norma"].isin(["Decreto Supremo N.° 007-2025-PCM", "Decreto Supremo N.° 008-2025-PCM"])
df.loc[corregidas, ["numero_norma", "titulo_completo", "es_lluvia", "tipo", "motivo"]]

es_lluvia
False    35
True     18
Name: count, dtype: int64
tipo
prorroga    37
declara     16
Name: count, dtype: int64
motivo
otro                 31
impacto de daños     17
peligro inminente     5
Name: count, dtype: int64


,numero_norma,titulo_completo,es_lluvia,tipo,motivo
5,Decreto Supremo N.° 008-2025-PCM,"Decreto Supremo que declara el Estado de Emergencia en los distritos de Chiclayo, José Leonardo Ortiz y La Victoria de la provincia de Chiclayo del departamento de Lambayeque, por peligro inminente ante colapso del sistema de alcantarillado",False,declara,peligro inminente
6,Decreto Supremo N.° 007-2025-PCM,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Loreto, Moquegua, Pasco, Puno, San Martín, Tacna y la Provincia Constitucional del Callao, por peligro inminente ante intensas precipitaciones pluviales",True,declara,peligro inminente


**11. Solo normas de lluvias**

In [ ]:
# 11. Solo normas de lluvias

PD.set_option("display.max_colwidth", None)

df_lluvias = df[df["es_lluvia"]].reset_index(drop=True)

print("Normas de lluvias:", len(df_lluvias))
df_lluvias[["numero_norma", "tipo", "motivo", "titulo_completo"]].head(5)



Normas de lluvias: 18


,numero_norma,tipo,motivo,titulo_completo
0,Decreto Supremo N.° 007-2025-PCM,declara,peligro inminente,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Loreto, Moquegua, Pasco, Puno, San Martín, Tacna y la Provincia Constitucional del Callao, por peligro inminente ante intensas precipitaciones pluviales"
1,Decreto Supremo N.° 024-2025-PCM,declara,impacto de daños,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, La Libertad, Lambayeque, Lima, Moquegua, Pasco, Piura, Puno, San Martín, Tacna, Tumbes y Ucayali, por impacto de daños a consecuencia de intensas precipitaciones pluviales."
2,Decreto Supremo N.° 021-2025-PCM,declara,impacto de daños,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Loreto, Madre de Dios, Piura, Puno, San Martín, Tacna y Ucayali, por impacto de daños a consecuencia de intensas precipitaciones pluviales"
3,Decreto Supremo N.° 034-2025-PCM,declara,impacto de daños,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lambayeque, Lima, Loreto, Moquegua, Pasco, Piura, San Martín, Tacna y Ucayali, por impacto de daños a consecuencia de intensas precipitaciones pluviales"
4,Decreto Supremo N.° 033-2025-PCM,declara,peligro inminente,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lambayeque, Lima, Loreto, Moquegua, Pasco, Piura, Puno, San Martín, Tacna y de la Provincia Constitucional del Callao, por peligro inminente ante intensas precipitaciones pluviales"


In [35]:
print(df_lluvias["tipo"].value_counts())

tipo
declara     11
prorroga     7
Name: count, dtype: int64


**12. Departamentos incluidos en cada decreto**


In [36]:
# 12. Departamentos incluidos en cada decreto

PD.set_option("display.max_colwidth", None)

departamentos = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
"Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
"Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
"Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
"San Martín", "Tacna", "Tumbes", "Ucayali"]

def departamentos_incluidos(titulo):
    encontrados = []
    for departamento in departamentos:
        if re.search(r"\b" + departamento + r"\b", titulo):
            encontrados.append(departamento)
    return encontrados


df_lluvias["departamentos"] = df_lluvias["titulo_completo"].apply(departamentos_incluidos)
df_lluvias["n_departamentos"] = df_lluvias["departamentos"].str.len()
df_lluvias[["numero_norma", "tipo", "n_departamentos", "departamentos"]]


,numero_norma,tipo,n_departamentos,departamentos
0,Decreto Supremo N.° 007-2025-PCM,declara,20,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Callao, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Loreto, Moquegua, Pasco, Puno, San Martín, Tacna]"
1,Decreto Supremo N.° 024-2025-PCM,declara,21,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, La Libertad, Lambayeque, Lima, Moquegua, Pasco, Piura, Puno, San Martín, Tacna, Tumbes, Ucayali]"
2,Decreto Supremo N.° 021-2025-PCM,declara,20,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Loreto, Madre de Dios, Piura, Puno, San Martín, Tacna, Ucayali]"
3,Decreto Supremo N.° 034-2025-PCM,declara,21,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lambayeque, Lima, Loreto, Moquegua, Pasco, Piura, San Martín, Tacna, Ucayali]"
4,Decreto Supremo N.° 033-2025-PCM,declara,22,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Callao, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lambayeque, Lima, Loreto, Moquegua, Pasco, Piura, Puno, San Martín, Tacna]"
5,Decreto Supremo N.° 026-2025-PCM,declara,18,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Moquegua, Pasco, Piura, Tacna, Tumbes, Ucayali]"
6,Decreto Supremo N.° 057-2025-PCM,declara,1,[Loreto]
7,Decreto Supremo N.° 056-2025-PCM,prorroga,7,"[Áncash, Cusco, Huánuco, La Libertad, Pasco, Piura, Tumbes]"
8,Decreto Supremo N.° 053-2025-PCM,prorroga,6,"[Amazonas, Áncash, Cusco, Piura, Tumbes, Ucayali]"
9,Decreto Supremo N.° 052-2025-PCM,declara,2,"[Cusco, Madre de Dios]"


**¿Cómo evitamos contar mal los departamentos?**

Buscamos cada departamento como una palabra completa en el título original.
 Así evitamos errores, por ejemplo, contar Ica dentro de Huancavelica.

Además, cada departamento se revisa solo una vez por decreto.
 Si aparece varias veces en el título, se cuenta una sola vez.

In [39]:
import os

os.makedirs("datos", exist_ok=True)
print("Carpeta datos lista")

Carpeta datos lista


**13. Guardar resultados en un archivo CSV**

In [38]:
# 13. Guardar resultados en un archivo CSV
# Instrucciones: 

# 13.1 Guardar los datos de los decretos por lluvias en 
# datos/decretos_lluvias.csv con las columnas numero, fecha, titulo, tipo, motivo y enlace.

decretos_lluvias = df_lluvias[["numero_norma", "fecha_publicacion", "titulo_completo",
                               "tipo", "motivo", "enlace_norma"]]

decretos_lluvias = decretos_lluvias.rename(columns={
    "numero_norma": "numero",
    "fecha_publicacion": "fecha",
    "titulo_completo": "titulo",
    "enlace_norma": "enlace",
})

decretos_lluvias.to_csv("datos/decretos_lluvias.csv", index=False, encoding="utf-8-sig")
print("Decretos guardados:", len(decretos_lluvias))


# 13.2 Guardar los datos de los decretos por departamento en 
# datos/decretos_por_departamento.csv con las columnas departamento, declaratorias y prorrogas.

filas = []

for departamento in departamentos:                  # recorre los 25 departamentos
    declaratorias = 0
    prorrogas = 0

    for i in range(len(df_lluvias)):                # recorre los 17 decretos
        if departamento in df_lluvias.loc[i, "departamentos"]:
            if df_lluvias.loc[i, "tipo"] == "declara":
                declaratorias = declaratorias + 1
            elif df_lluvias.loc[i, "tipo"] == "prorroga":
                prorrogas = prorrogas + 1

    filas.append({"departamento": departamento,
                  "declaratorias": declaratorias,
                  "prorrogas": prorrogas})

decretos_por_departamento = PD.DataFrame(filas)

decretos_por_departamento.to_csv("datos/decretos_por_departamento.csv", index=False, encoding="utf-8-sig")
decretos_por_departamento




Decretos guardados: 18


,departamento,declaratorias,prorrogas
0,Amazonas,8,4
1,Áncash,8,6
2,Apurímac,7,2
3,Arequipa,8,2
4,Ayacucho,8,4
5,Cajamarca,7,2
6,Callao,2,0
7,Cusco,9,6
8,Huancavelica,8,2
9,Huánuco,8,5
